# 중첩(Superposition): 차원보다 많은 정보 표현 - 중첩 현상 이해

- Tutorial ID: `tut-12-1`
- Tutorial: 중첩(Superposition): 차원보다 많은 정보 표현
- Section ID: `s12-1-1`
- Section: 중첩 현상 이해


In [ ]:
# ============================================================
# 코드 읽는 법 — 중첩 현상 이해
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 55)
print("중첩(Superposition) 현상 시뮬레이션")
print("=" * 55)

np.random.seed(42)

In [ ]:
# 1. 중첩 없는 경우: n차원에 n개 특징

In [ ]:
print("\n1. 중첩 없는 경우 (직교 표현)")
print("-" * 40)

n_dims = 4
n_features_no_superpos = 4

# 직교 기저로 특징 저장
W_no_superpos = np.eye(n_dims)  # 각 특징이 독립 차원 사용
print(f"  {n_dims}차원에 {n_features_no_superpos}개 특징 저장")
print(f"  W: {W_no_superpos.shape}")
print("  특징 벡터 (직교):")
for i in range(n_features_no_superpos):
    print(f"    특징 {i}: {W_no_superpos[i]}")

# 재구성 오류 = 0 (완벽)
x = np.array([1., 0., 1., 0.])  # 특징 0, 2 활성화
h = W_no_superpos @ x
recon = W_no_superpos.T @ h
error = np.linalg.norm(x - recon)
print(f"  재구성 오류: {error:.6f} (완벽!)")

In [ ]:
# 2. 중첩 있는 경우: n차원에 n보다 많은 특징

In [ ]:
print("\n2. 중첩 있는 경우 (비직교 표현)")
print("-" * 40)

n_features_superpos = 8  # 차원보다 2배 많은 특징!

# 정20면체 등 준직교 방향 사용
angles = np.linspace(0, np.pi * (1 - 1/n_features_superpos), n_features_superpos)
W_superpos = np.zeros((n_features_superpos, n_dims))
for i in range(n_features_superpos):
    # 비직교 방향들
        W_superpos[i, 0] = np.cos(angles[i])
        W_superpos[i, 1] = np.sin(angles[i])
        W_superpos[i, 2] = np.cos(angles[i] * 1.7) * 0.5
        W_superpos[i, 3] = np.sin(angles[i] * 1.7) * 0.5
        W_superpos[i] /= np.linalg.norm(W_superpos[i])

print(f"  {n_dims}차원에 {n_features_superpos}개 특징 저장 ({n_features_superpos/n_dims:.1f}x 중첩)")

# 내적 (간섭 정도)
gram = W_superpos @ W_superpos.T
np.fill_diagonal(gram, 0)
max_interference = np.max(np.abs(gram))
avg_interference = np.mean(np.abs(gram))
print(f"  특징 간 최대 간섭: {max_interference:.3f}")
print(f"  특징 간 평균 간섭: {avg_interference:.3f}")

# 희소 입력으로 재구성 테스트
# 한 번에 하나의 특징만 활성화 (희소!)
test_feature = 3
x_sparse = np.zeros(n_features_superpos)
x_sparse[test_feature] = 1.0

h_sparse = W_superpos.T @ x_sparse  # 인코딩
recon_sparse = W_superpos @ h_sparse  # 디코딩 (근사)

print(f"\n  희소 특징 재구성 테스트 (특징 {test_feature}):")
print(f"  재구성된 활성화: {np.round(recon_sparse, 3)}")
print(f"  타겟: {np.round(x_sparse[:5], 3)}...")
print(f"  재구성 오류: {np.linalg.norm(x_sparse - recon_sparse):.4f}")

In [ ]:
# 3. 희소성이 중첩을 가능하게 한다

In [ ]:
print("\n3. 희소성 vs 간섭 트레이드오프")
print("-" * 40)

for sparsity in [0.9, 0.7, 0.5, 0.3, 0.1]:
    # 각 특징이 sparsity 확률로 0인 경우
    n_trials = 100
    errors = []
        for _ in range(n_trials):
        x_rand = (np.random.rand(n_features_superpos) > sparsity).astype(float)
        if np.sum(x_rand) == 0:
            continue
        h_rand = W_superpos.T @ x_rand
        recon_rand = W_superpos @ h_rand
        # 양수 클리핑 (ReLU 근사)
        recon_clip = np.maximum(0, recon_rand)
        errors.append(np.linalg.norm(x_rand - recon_clip))
    
    avg_err = np.mean(errors) if errors else float('inf')
    bar = '█' * int((1 - min(avg_err, 1)) * 15)
    print(f"  희소도 {sparsity:.1f}: 평균 오류={avg_err:.3f} 정확도: {bar}")

print("\n  → 희소도가 높을수록 (대부분 0) 중첩이 더 잘 작동!")
print("  → 트랜스포머의 MLP 뉴런은 매우 희소하게 활성화됨")
print("  → 이것이 초과 구독(oversubscription)을 가능하게 함")

print("\n⚠️ 중첩은 기계론적 해석성을 어렵게 만듭니다:")
print("  개별 뉴런이나 방향이 단일 개념을 나타내지 않을 수 있음")
print("  → 특징은 '방향'으로 표현되며 뉴런과 1:1 대응하지 않음")